# 06c · `aede` en grilla de anchos, pesos y maxcl (vigente: niveles, 16 series)

**Objetivo.** Correr la EPCA del asesor (`aede`, port de Python `src/camiz_port.py`, D6) sobre la
grilla declarada de antemano en D10 (`decisiones.qmd`):

- anchos $n_w$ ∈ {24, 36, 48, 60} meses, paso de un mes, tres dimensiones (D9);
- pesos uniformes y gaussianos (`nsd = 3`);
- Fisher sobre λ1 con maxcl ∈ {10, 15, 20} (D17.1: maxcl = 10 es la especificación principal);
- criterios: codo por caída máxima y Calinski-Harabász con $k$ y con $k-1$ grados de libertad
  (D11: el CH se informa y no selecciona).

La grilla se lee como **multiescala**, no como búsqueda de la ventana «correcta» (D10), y cada
$k^*$ se lee contra el nulo de estructura constante (notebook 07b), no contra los episodios.

**Parámetros** (`src/parametros.py`): por defecto, niveles, 16 series (D16) y salidas en
`data/results/final/`. Con `TESIS_VIA=ambas` y `TESIS_EXCLUIDAS=[]` reproduce el antecedente
(17 series, dos vías): prueba de no regresión `herramientas/no_regresion_dinamica.py`.

**Paridad (D6).** La referencia es el Rmd final (sección 4), que escribe `R_aede_cortes.csv`; este
notebook lo compara con el suyo y falla si difieren. Con 17 series, las filas `nw = 60`,
`maxcl = 10` deben ser además las de `999_port_aede_cortes.csv` (iteración 3a).

**Convención de fechado.** La fecha de cada ventana es la de su fila central, $i + n_w/2$
(1-based), como `rownames(serie)[(1 + nw/2):(wn + nw/2)]` en `aede_r`.

**Semillas.** `aede` y Fisher son determinísticos. El único sorteo es el del umbral de Horn de la
sección 2 (D5): `SEMILLA = 2026`, con el generador PCG64 de numpy, informativo; los conteos oficiales
usan el umbral que escribe el Rmd (`R_aede_horn_umbrales.csv`).

**Salidas:** `06c_aede_cortes.csv`, `06c_aede_valores_propios.csv`,
`06c_aede_valores_propios_resumen.csv` y `06c_aede_cruces.csv`.

In [1]:
# ---- Configuración y parámetros (adenda 3b, §3) ------------------------------------
import sys
from pathlib import Path

import numpy as np
import pandas as pd

RAIZ = Path.cwd().resolve()
if not (RAIZ / '_quarto.yml').exists():
    RAIZ = RAIZ.parent
assert (RAIZ / '_quarto.yml').exists(), 'No se encontró la raíz del repositorio'
sys.path.insert(0, str(RAIZ))
from src.parametros import leer_parametros
from src.datos import cargar_vias, meta_vigente

PARAM = leer_parametros(RAIZ)
VIA, EXCLUIDAS, DIR, DIR_R = PARAM['VIA'], PARAM['EXCLUIDAS'], PARAM['DIR'], PARAM['DIR_R']
ANTECEDENTE = len(EXCLUIDAS) == 0          # 17 series: modo de no regresión
X, Xt = cargar_vias(RAIZ, excluidas=EXCLUIDAS)
META = meta_vigente(EXCLUIDAS)
P = X.shape[1]
TODAS = {'niveles': X, 'transformadas': Xt}
VIAS = {v: (TODAS[v], DIR[v], DIR_R[v]) for v in PARAM['VIAS_ACTIVAS']}
print('VIA =', VIA, '| EXCLUIDAS =', EXCLUIDAS, '| p =', P, '| CARPETA =', PARAM['CARPETA'])
for via, (M, d, dr) in VIAS.items():
    print(via, M.shape, M.index[0], '→', M.index[-1], '| salidas:', d.relative_to(RAIZ) if d.is_relative_to(RAIZ) else d)

from src import camiz_port as cp

SEMILLA = 2026
NW_GRILLA = (24, 36, 48, 60)
PESOS = {'uniformes': 0, 'gaussianos_nsd3': 3}
MAXCL_GRILLA = (10, 15, 20)
NDIM = 3
CRITERIOS = [('codo (caída máxima)', 'k_codo', 'cortes_codo'),
             ('Calinski-Harabász k gl (aede)', 'k_ch', 'cortes_ch'),
             ('Calinski-Harabász k-1 gl (habitual)', 'k_ch1', 'cortes_ch1')]

VIA = niveles | EXCLUIDAS = ('Liquidez M2',) | p = 16 | CARPETA = /home/claude/tesis-modelamiento-matematico/data/results/final
niveles (228, 16) 2007-01 → 2025-12 | salidas: data/results/final


## 1. Grilla de `aede` (D4)

In [2]:
corridas = {}      # (vía, nw, pesos) -> salida de aede con maxcl = 10 (para la sección 2)
cortes = {}
for via, (M, d, dr) in VIAS.items():
    filas = []
    for nw in NW_GRILLA:
        for pesos, nsd in PESOS.items():
            for maxcl in MAXCL_GRILLA:
                a = cp.aede(M, nw=nw, ndim=NDIM, nsd=nsd, maxcl=maxcl)
                r = cp.resumen_aede(a)
                if maxcl == MAXCL_GRILLA[0]:
                    corridas[(via, nw, pesos)] = a
                for crit, kk, cc in CRITERIOS:
                    filas.append({'nw': nw, 'maxcl': maxcl, 'pesos': pesos, 'criterio': crit, 'k': r[kk],
                                  'k_en_tope': r[kk] == maxcl, 'cortes': ' '.join(r[cc])})
    tab = pd.DataFrame(filas)
    tab.to_csv(d / '06c_aede_cortes.csv', index=False)
    cortes[via] = tab
    print(via, len(tab), 'filas')

niveles 72 filas


### 1.1 Comparación con R (y, con 17 series, con la iteración 3a)

In [3]:
for via, (M, d, dr) in VIAS.items():
    py = pd.read_csv(d / '06c_aede_cortes.csv', keep_default_na=False)
    f3a = dr / '999_port_aede_cortes.csv'
    if ANTECEDENTE and f3a.exists():
        viejo = pd.read_csv(f3a, keep_default_na=False)
        s60 = py[(py.nw == 60) & (py.maxcl == 10)].reset_index(drop=True)
        assert (s60[['pesos', 'criterio', 'k', 'cortes']].astype(str).to_numpy()
                == viejo[['pesos', 'criterio', 'k', 'cortes']].astype(str).to_numpy()).all(), via
        print(f'{via}: reproduce las filas de 3a ({f3a.name})')
    f_r = dr / 'R_aede_cortes.csv'
    if f_r.exists():
        ref = pd.read_csv(f_r, keep_default_na=False)
        clave = lambda t: [f'{a} {b} {c} {e} {k} {str(t_).upper()} {ct}' for a, b, c, e, k, t_, ct in
                           zip(t.nw, t.maxcl, t.pesos, t.criterio, t.k, t.k_en_tope, t.cortes)]
        n_dif = sum(a != b for a, b in zip(clave(ref), clave(py)))
        assert len(ref) == len(py) and n_dif == 0, (via, n_dif)
        print(f'{via}: coincide con {f_r.name} ({len(py)} filas)')
    else:
        print(f'{via}: aún no existe {f_r.name} (ejecutar el Rmd)')

niveles: coincide con R_aede_cortes.csv (72 filas)


In [4]:
# Vista compacta: k por criterio y maxcl (* = en el tope)
for via in VIAS:
    t = cortes[via].copy()
    t['celda'] = t.k.astype(str) + np.where(t.k_en_tope, '*', '')
    print(via)
    display(t.pivot_table(index=['nw', 'pesos', 'criterio'], columns='maxcl', values='celda', aggfunc='first'))

niveles


maxcl                                                    10   15   20
nw pesos           criterio                                          
24 gaussianos_nsd3 Calinski-Harabász k gl (aede)        10*  15*  20*
                   Calinski-Harabász k-1 gl (habitual)  10*  15*  20*
                   codo (caída máxima)                    2    2    2
   uniformes       Calinski-Harabász k gl (aede)        10*  15*  20*
                   Calinski-Harabász k-1 gl (habitual)    6  15*  20*
                   codo (caída máxima)                    6    6    6
36 gaussianos_nsd3 Calinski-Harabász k gl (aede)        10*  15*  20*
                   Calinski-Harabász k-1 gl (habitual)    2  15*  20*
                   codo (caída máxima)                    2    2    2
   uniformes       Calinski-Harabász k gl (aede)        10*  15*  20*
                   Calinski-Harabász k-1 gl (habitual)    2    2  20*
                   codo (caída máxima)                    2    2    2
48 gaussianos_nsd3 Calinski-Harabász k gl (aede)          3  15*  20*
                   Calinski-Harabász k-1 gl (habitual)    2    2    2
                   codo (caída máxima)                    2    2    2
   uniformes       Calinski-Harabász k gl (aede)        10*  15*  20*
                   Calinski-Harabász k-1 gl (habitual)    2  15*  20*
                   codo (caída máxima)                    2    2    2
60 gaussianos_nsd3 Calinski-Harabász k gl (aede)        10*  15*  20*
                   Calinski-Harabász k-1 gl (habitual)  10*  15*  20*
                   codo (caída máxima)                    2    2    2
   uniformes       Calinski-Harabász k gl (aede)        10*  15*  20*
                   Calinski-Harabász k-1 gl (habitual)    2    2  20*
                   codo (caída máxima)                    2    2    2

## 2. Valores propios por ventana frente a Kaiser y Horn (D5; apunte 7 en su versión útil, D14)

Para cada ancho y esquema de pesos: λ1–λ3 de cada ventana; el $n$ efectivo de Kish de los pesos,
$n_\omega = 1/\sum_i \omega_i^2$ (pesos normalizados a suma 1), que depende solo de los pesos; el
umbral de Kaiser (1) y el p95 de Horn para λ1–λ3 con $n = \lceil n_\omega \rceil$ (1 000
simulaciones gaussianas de $p$ series). El techo se toma con una holgura de $10^{-9}$.

**Cruce λ2 ↔ λ3:** la ventana $i$ registra un cruce si su eje 2 es más congruente (|Tucker|) con
el eje 3 de la ventana $i-1$ que con el eje 2 (cargas alineadas de `aede`, `correlationlsq`).

**Umbral de Horn compartido.** Los conteos oficiales usan el umbral que escribe el Rmd
(`R_aede_horn_umbrales.csv`); el de numpy y su conteo se informan aparte (`horn_p95_numpy`,
`bajo_horn_numpy`). **Empates con Kaiser:** «bajo Kaiser» si $\lambda < 1 - 10^{-9}$; las ventanas
a menos de $10^{-9}$ de 1 se cuentan aparte (`en_umbral_kaiser`).

In [5]:
from src.multivariado import umbral_horn, n_kish, tucker

TOL_KAISER = 1e-9
cache_horn = {}
def horn_numpy(n_h):
    if n_h not in cache_horn:
        cache_horn[n_h] = umbral_horn(n_h, P, n_sim=1000, percentil=0.95, semilla=SEMILLA)
    return cache_horn[n_h]

f_horn = DIR_R['niveles'] / 'R_aede_horn_umbrales.csv'
HORN_R = pd.read_csv(f_horn) if f_horn.exists() else None
def horn_compartido(n_h):
    # Umbral p95 de R para λ1..λ3 con n_h observaciones; si no existe, el de numpy
    if HORN_R is not None and (HORN_R.n_horn == n_h).any():
        return HORN_R[HORN_R.n_horn == n_h].sort_values('eje').horn_p95.to_numpy()[:3], 'R'
    return horn_numpy(n_h)[:3], 'numpy'

def constante_ventana(M, nw):
    A = M.to_numpy()
    return np.array([(A[i:i + nw] == A[i]).all(axis=0).any() for i in range(len(A) - nw + 1)])

vpv = {}
for via, (M, d, dr) in VIAS.items():
    vp_l, res_l, cr_l = [], [], []
    for nw in NW_GRILLA:
        for pesos in PESOS:
            a = corridas[(via, nw, pesos)]
            ev = a['eigenvalues'].to_numpy()
            n_om = n_kish(a['weights'])
            n_h = int(np.ceil(n_om - 1e-9))
            hu, fuente = horn_compartido(n_h)
            hn = horn_numpy(n_h)[:3]
            # Criterio 5 del plan (adenda: λ1 + λ2 + λ3 <= p) y n_omega = nw con pesos uniformes
            assert (ev.sum(axis=1) <= P + 1e-9).all()
            assert pesos != 'uniformes' or abs(n_om - nw) < 1e-9
            fv = list(a['eigenvalues'].index)
            vp_l.append(pd.DataFrame({'fecha': fv, 'nw': nw, 'pesos': pesos, 'lambda1': ev[:, 0], 'lambda2': ev[:, 1],
                                      'lambda3': ev[:, 2], 'n_omega': n_om, 'n_horn': n_h, 'umbral_kaiser': 1.0,
                                      'horn_p95_1': hu[0], 'horn_p95_2': hu[1], 'horn_p95_3': hu[2],
                                      'horn_p95_numpy_1': hn[0], 'horn_p95_numpy_2': hn[1], 'horn_p95_numpy_3': hn[2]}))
            C = a['correlationlsq']
            t22 = np.array([abs(tucker(C[i, :, 1], C[i - 1, :, 1])) for i in range(1, len(ev))])
            t23 = np.array([abs(tucker(C[i, :, 1], C[i - 1, :, 2])) for i in range(1, len(ev))])
            cruce = np.nonzero(t23 > t22)[0] + 1
            cr_l.append(pd.DataFrame({'fecha': [fv[i] for i in cruce], 'nw': nw, 'pesos': pesos,
                                      'tucker_22': t22[cruce - 1], 'tucker_23': t23[cruce - 1],
                                      'brecha_23': ev[cruce, 1] - ev[cruce, 2]}))
            res_l.append(pd.DataFrame({'nw': nw, 'pesos': pesos, 'eje': [1, 2, 3], 'ventanas': len(ev), 'n_omega': n_om,
                                       'n_horn': n_h, 'bajo_kaiser': (ev < 1 - TOL_KAISER).sum(axis=0),
                                       'en_umbral_kaiser': (np.abs(ev - 1) <= TOL_KAISER).sum(axis=0),
                                       'serie_constante': int(constante_ventana(M, nw).sum()),
                                       'bajo_horn': (ev < hu).sum(axis=0), 'horn_p95': hu, 'fuente_umbral_horn': fuente,
                                       'bajo_horn_numpy': (ev < hn).sum(axis=0), 'horn_p95_numpy': hn,
                                       'cruces_2_3': len(cruce)}))
    v = {'vp': pd.concat(vp_l, ignore_index=True), 'resumen': pd.concat(res_l, ignore_index=True),
         'cruces': pd.concat(cr_l, ignore_index=True)}
    v['vp'].to_csv(d / '06c_aede_valores_propios.csv', index=False)
    v['resumen'].to_csv(d / '06c_aede_valores_propios_resumen.csv', index=False)
    v['cruces'].to_csv(d / '06c_aede_cruces.csv', index=False)
    vpv[via] = v
    print(via, len(v['vp']), 'ventanas ·', len(v['cruces']), 'cruces λ2 ↔ λ3')
    display(v['resumen'])

niveles 1496 ventanas · 19 cruces λ2 ↔ λ3


,nw,pesos,eje,ventanas,n_omega,n_horn,bajo_kaiser,en_umbral_kaiser,serie_constante,bajo_horn,horn_p95,fuente_umbral_horn,bajo_horn_numpy,horn_p95_numpy,cruces_2_3
0,24,uniformes,1,205,24.000000,24,0,0,7,0,3.178180,R,0,3.176183,3
1,24,uniformes,2,205,24.000000,24,0,0,7,117,2.565380,R,121,2.593557,3
2,24,uniformes,3,205,24.000000,24,76,7,7,199,2.159624,R,199,2.185039,3
3,24,gaussianos_nsd3,1,205,18.237952,19,0,0,7,0,3.545886,R,0,3.559213,5
4,24,gaussianos_nsd3,2,205,18.237952,19,0,0,7,126,2.827941,R,124,2.796669,5
5,24,gaussianos_nsd3,3,205,18.237952,19,51,6,7,199,2.339064,R,199,2.333832,5
6,36,uniformes,1,193,36.000000,36,0,0,0,0,2.699826,R,0,2.690452,3
7,36,uniformes,2,193,36.000000,36,1,0,0,52,2.236296,R,52,2.247664,3
8,36,uniformes,3,193,36.000000,36,55,0,0,179,1.946385,R,179,1.949295,3
9,36,gaussianos_nsd3,1,193,27.626910,28,0,0,0,0,2.968820,R,0,2.979417,1


### 2.1 Comparación con R

λ1–λ3 y $n_\omega$ con tolerancia $10^{-10}$; conteos bajo Kaiser, cruces y conteos bajo Horn
(con el umbral compartido) exactos.

In [6]:
for via, (M, d, dr) in VIAS.items():
    f_r = dr / 'R_aede_valores_propios.csv'
    if not f_r.exists():
        print(f'{via}: aún no existe {f_r.name} (ejecutar el Rmd)')
        continue
    v = vpv[via]
    ref = pd.read_csv(f_r)
    m = v['vp'].merge(ref, on=['nw', 'pesos', 'fecha'], suffixes=('', '_r'), validate='one_to_one')
    assert len(m) == len(ref) == len(v['vp'])
    dif = max(float(np.max(np.abs(m[c] - m[c + '_r']))) for c in ['lambda1', 'lambda2', 'lambda3', 'n_omega'])
    assert dif < 1e-10, (via, dif)
    rr = pd.read_csv(dr / 'R_aede_valores_propios_resumen.csv')
    mr = v['resumen'].merge(rr, on=['nw', 'pesos', 'eje'], suffixes=('', '_r'))
    assert (v['resumen'].fuente_umbral_horn == 'R').all(), 'umbral de Horn sin compartir: ejecutar primero el Rmd'
    for c in ['ventanas', 'n_horn', 'bajo_kaiser', 'en_umbral_kaiser', 'serie_constante', 'cruces_2_3', 'bajo_horn']:
        assert (mr[c] == mr[c + '_r']).all(), (via, c, mr.loc[mr[c] != mr[c + '_r'], ['nw', 'pesos', 'eje', c, c + '_r']])
    dif_h = float(np.max(np.abs(mr.horn_p95_numpy - mr.horn_p95_r)))
    n_dif_numpy = int((mr.bajo_horn_numpy != mr.bajo_horn).sum())
    cr = pd.read_csv(dr / 'R_aede_cruces.csv')
    assert list(zip(cr.nw, cr.pesos, cr.fecha)) == list(zip(v['cruces'].nw, v['cruces'].pesos, v['cruces'].fecha)), via
    print(f'{via}: coincide con R (dif. máx. λ y n_omega {dif:.1e}; conteos y cruces exactos; '
          f'umbral de Horn compartido; el de numpy difiere hasta {dif_h:.3f} y cambia el conteo en {n_dif_numpy} de {len(mr)} filas)')

niveles: coincide con R (dif. máx. λ y n_omega 7.8e-14; conteos y cruces exactos; umbral de Horn compartido; el de numpy difiere hasta 0.031 y cambia el conteo en 4 de 24 filas)


**Lectura.** Este notebook no interpreta (regla 7): la grilla de cortes y los valores propios por
ventana se leen contra el nulo de estructura constante del notebook 07b (D10, D11, D13 y D17).